# 🎯 Chapter 05 — Model Optimization & Tuning

## 📖 Objective
Optimize Random Forest performance by tuning key hyperparameters and analyzing trade-offs between model complexity and predictive accuracy.

### ⚙️ Step 01 — Notebook Metadata
Define notebook metadata and objectives for model tuning.

In [12]:
# Goal   : Optimize Random Forest model using GridSearchCV & RandomizedSearchCV
# ================================================================
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import PROCESSED_DIR, MODEL_DIR, OUTPUT_DIR, FIGURE_DIR, PREDICTION_DIR, RESULT_DIR
from src.utils.emoji_log import info, success, save

# ------------------------------------------------
# Notebook Info
# ------------------------------------------------
info("Chapter 05 initialized — Ready for model tuning.")
success(f"Data directory: {PROCESSED_DIR}")
success(f"Model directory: {MODEL_DIR}")
success(f"Results directory: {OUTPUT_DIR}")

💬 Chapter 05 initialized — Ready for model tuning.
✅ Data directory: C:\Users\dinni\OneDrive\桌面\air_pollution\data\processed
✅ Model directory: C:\Users\dinni\OneDrive\桌面\air_pollution\models
✅ Results directory: C:\Users\dinni\OneDrive\桌面\air_pollution\output


### 🧩 Step 02 — Parameter Grid Design
Design search grids for `n_estimators`, `max_depth`, `max_features`, and `min_samples_split`.

In [2]:
from pprint import pprint

In [3]:
# ------------------------------------------------
# Parameter ranges (coarse search for RandomizedSearchCV)
# ------------------------------------------------
param_distributions = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 10, 20, 30, 40, 50],
    "max_features": ["auto", "sqrt", "log2"],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False],
}

info("Parameter distributions for RandomizedSearchCV:")
pprint(param_distributions)

💬 Parameter distributions for RandomizedSearchCV:
{'bootstrap': [True, False],
 'max_depth': [None, 10, 20, 30, 40, 50],
 'max_features': ['auto', 'sqrt', 'log2'],
 'min_samples_leaf': [1, 2, 4],
 'min_samples_split': [2, 5, 10],
 'n_estimators': [100, 200, 300, 400, 500]}


In [4]:
# ------------------------------------------------
# Narrow grid for fine-tuning
# ------------------------------------------------
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

info("Parameter grid for GridSearchCV:")
pprint(param_grid)

💬 Parameter grid for GridSearchCV:
{'max_depth': [10, 20, 30],
 'min_samples_leaf': [1, 2],
 'min_samples_split': [2, 5],
 'n_estimators': [200, 300, 400]}


### 🔍 Step 03 — RandomizedSearchCV Setup
Set up and run Randomized Search to identify promising parameter regions efficiently.

In [5]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.modeling.train_baseline import (
    split_train_test,
    load_cleaned_data,
    build_features,
)
from src.features.feature_engineering import (
    clip_pollutants,
    add_rolling_features,
    handle_outliers_iqr,
    log_transform_features,
    scale_features,
)
import numpy as np
import time
import json
from datetime import datetime

In [6]:
filename = "Taiwan"
df = load_cleaned_data(filename)

# === Feature Engineering Pipeline ===
df_clip = clip_pollutants(df.copy())
df_rolling = add_rolling_features(df_clip)
df_iqr = handle_outliers_iqr(df_rolling)
df_log = log_transform_features(df_iqr)

# === Feature Selection & Scaling ===
X, y = build_features(df_log)
X_scaled = scale_features(X, save_=False)
data_split = split_train_test(X_scaled, y)

# ------------------------------------------------
# 1️⃣ Load data (assume data_split from Chapter 04)
# ------------------------------------------------
X_train, X_test = data_split["X_train"], data_split["X_test"]
y_train, y_test = data_split["y_train"], data_split["y_test"]

📂 Loading CSV: C:\Users\dinni\OneDrive\桌面\air_pollution\data\processed\Taiwan.csv
✅ Read CSV successfully! Shape: (5823862, 25)
✅ The pollutants limit has been set.
⚠️ Skipping nox due to NO + NO2 already exist.
✅ Rolling features added.
✅ IQR has been set.
✅ Pollutants skewes has been smoothed.
💬 Features shape: (5823862, 32), Target shape: (5823862,)
✅ Numerical features have been standardized. (32 columns scaled)
✅ Data successfully split! Train: (4659089, 32), Test: (1164773, 32)


In [7]:
# ------------------------------------------------
# 2️⃣ Initialize baseline model
# ------------------------------------------------
rf_model = RandomForestRegressor(random_state=42, n_jobs=4)

In [ ]:
# ------------------------------------------------
# 3️⃣ Set up Randomized Search
# ------------------------------------------------
n_iter_search = 5 # number of random parameter sets to try
cv_folds = 2 # cross-validation folds

random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_iter=n_iter_search,
    cv=cv_folds,
    scoring="neg_mean_absolute_error",
    n_jobs=4,
    verbose=2,
    random_state=42
)

X_sample = X_train.sample(frac=0.05, random_state=42)
y_sample = y_train.loc[X_sample.index]

In [ ]:
# ------------------------------------------------
# 4️⃣ Run search and record execution time
# ------------------------------------------------
start_time = time.time()
random_search.fit(X_sample, y_sample)
elapsed_time = time.time() - start_time

success(f"RandomizedSearchCV completed in {elapsed_time:.2f} seconds")

Fitting 2 folds for each of 5 candidates, totalling 10 fits
✅ RandomizedSearchCV completed in 676.88 seconds


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
model_path = MODEL_DIR / f"RandomForest_RandomSearch_{timestamp}.pkl"
params_path = RESULT_DIR / f"best_params_{timestamp}.json"
cv_results_path = RESULT_DIR / f"cv_results_{timestamp}.csv"

In [ ]:
joblib.dump(random_search.best_estimator_, model_path)
success(f"Model saved to: {model_path}")

✅ Model saved to: C:\Users\dinni\OneDrive\桌面\air_pollution\models\RandomForest_RandomSearch_20251113_1123.pkl


In [ ]:
with open(params_path, "w") as f:
    json.dump(random_search.best_params_, f, indent=4)
success(f"Best params saved to: {params_path}")

✅ Best params saved to: C:\Users\dinni\OneDrive\桌面\air_pollution\output\predictions\best_params_20251113_1123.json


In [ ]:
cv_result_df = pd.DataFrame(random_search.cv_results_)
cv_result_df.to_csv(cv_results_path)
success(f"CV results saved to: {cv_results_path}")

✅ CV results saved to: C:\Users\dinni\OneDrive\桌面\air_pollution\output\predictions\cv_results_20251113_1123.csv



### 🧮 Step 04 — GridSearchCV Refinement
Perform fine-grained Grid Search on top-performing parameter combinations.

In [7]:
random_search = joblib.load(MODEL_DIR / "RandomForest_RandomSearch_20251113_1123.pkl")

In [8]:
random_search

,n_estimators,500
,criterion,'squared_error'
,max_depth,40
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'log2'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [9]:
best_params_path = RESULT_DIR / "best_params_20251113_1123.json"
with open(best_params_path, "r") as f:
    best_params = json.load(f)
info("Best parameters from RandomizedSearchCV:")
pprint(best_params)

💬 Best parameters from RandomizedSearchCV:
{'bootstrap': False,
 'max_depth': 40,
 'max_features': 'log2',
 'min_samples_leaf': 2,
 'min_samples_split': 2,
 'n_estimators': 500}


In [10]:
# Fine-tuning
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [30, 40, 50],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 2],
    'max_features': ['log2'],
    'bootstrap': [False]
}

info("Fine-tuning parameter grid:")
pprint(param_grid)

💬 Fine-tuning parameter grid:
{'bootstrap': [False],
 'max_depth': [30, 40, 50],
 'max_features': ['log2'],
 'min_samples_leaf': [1, 2],
 'min_samples_split': [2, 3],
 'n_estimators': [200, 300, 400]}


In [11]:
# GridSearchCV
rf_refine = RandomForestRegressor(random_state=42, n_jobs=1)

grid_search = GridSearchCV(
    estimator=rf_refine,
    param_grid=param_grid,
    cv=2,
    scoring="neg_mean_absolute_error",
    n_jobs=4,
    verbose=2,
    return_train_score=False, # memory saver
    pre_dispatch="2*n_jobs"   # prevents stampede on Windows
)

# fraction sampling
X_refine = X_train.sample(frac=0.2, random_state=42)
y_refine = y_train.loc[X_refine.index]

start_time = time.time()
grid_search.fit(X_refine, y_refine)
elapsed_time = time.time() - start_time
success(f"GridSearchCV completed in {elapsed_time/60:.1f} min")

Fitting 2 folds for each of 36 candidates, totalling 72 fits


c:\Users\dinni\AppData\Local\pypoetry\Cache\virtualenvs\air-pollution-CdZUSHHx-py3.13\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
36 fits failed out of a total of 72.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\dinni\AppData\Local\pypoetry\Cache\virtualenvs\air-pollution-CdZUSHHx-py3.13\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\dinni\AppData\Local\pypoetry\Cache\virtualenvs\air-pollution-CdZUSHHx-py3.13\Lib\site-packages\sklearn\base.py", line 136

✅ GridSearchCV completed in 538.7 min


In [13]:
timestamp = time.strftime("%Y%m%d_%H%M")
model_path = MODEL_DIR / f"RandomForest_GridSearch_{timestamp}.pkl"
joblib.dump(grid_search.best_estimator_, model_path)
save(f"Model saved to: {model_path}")

💾 Model saved to: C:\Users\dinni\OneDrive\桌面\air_pollution\models\RandomForest_GridSearch_20251114_0856.pkl


In [14]:
params_path = RESULT_DIR / f"grid_best_params_{timestamp}.json"
with open(params_path, "w") as f:
    json.dump(grid_search.best_params_, f, indent=4)
save(f"Best params saved to: {params_path}")

💾 Best params saved to: C:\Users\dinni\OneDrive\桌面\air_pollution\result\grid_best_params_20251114_0856.json


In [15]:
cv_results_path = RESULT_DIR / f"grid_cv_results_{timestamp}.csv"
cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_results_df.to_csv(cv_results_path, index=False)
save(f"CV results saved to: {cv_results_path}")

💾 CV results saved to: C:\Users\dinni\OneDrive\桌面\air_pollution\result\grid_cv_results_20251114_0856.csv


In [16]:
grid_search.best_params_

{'bootstrap': False,
 'max_depth': 50,
 'max_features': 'log2',
 'min_samples_leaf': 2,
 'min_samples_split': 3,
 'n_estimators': 400}

In [17]:
grid_search.best_score_

np.float64(-6.370014494414683)

In [21]:
cv_results_df["mean_test_score"].isna().sum() / len(cv_results_df)

np.float64(0.7222222222222222)

### 📊 Step 05 — Evaluation and Comparison
Compare tuned Random Forest vs baseline RF using MAE, RMSE, and R² metrics.


In [ ]:
# ------------------------------------------------
# 1️⃣ Load three models
# ------------------------------------------------

### 📈 Step 06 — Visualize Search Results
Plot validation curves, parameter importance heatmaps, and learning curves.

### 🧠 Step 07 — Interpretation and Insights
Summarize tuning outcomes and discuss trade-offs (e.g., bias vs variance, complexity vs performance).


### 💾 Step 08 — Save Best Model and Metrics
Export best model (`RandomForest_Tuned.pkl`) and its performance summary.


### 🚀 Step 09 — Discussion and Next Chapter
Summarize optimization improvements and preview next chapter (e.g., Gradient Boosting or SHAP-based explainability).